# Workshop: S2S – Accessing Hexagons

**Duration:** 30 minutes  
**Type:** Hands-on exercise  
**Presenter:** Ben Stewart, Gabriel Levin

---

### What you'll learn

1. Why H3 hexagons and when to use them vs. admin boundaries
2. How to discover what data is available through the S2S API
3. How to query hex-level data for an area of interest
4. How to aggregate hex data back to admin boundaries
5. How to work with time-series data (drought/SPI)

### Key takeaway

> *Admin boundaries give you the headline number. Hexagons give you the spatial detail **within** a district*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://githubtocolab.com/worldbank/DECAT_Space2Stats/blob/main/docs/user-docs/space2stats_accessing_hexagons.ipynb)

In [ ]:
!pip install space2stats-client matplotlib mapclassify plotnine folium

In [ ]:
from space2stats_client import Space2StatsClient
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import json
import folium
from shapely import from_geojson
from shapely.geometry import shape

# Initialize the client — that’s it, no auth required
client = Space2StatsClient()

---
## Part 1: Why Hexagons? (5 min)

### The problem with admin boundaries

Admin boundaries are:
- **Irregular** — a tiny urban district vs. a massive rural province
- **Politically defined** — they change, and they don’t reflect physical reality
- **Incomparable** — you can’t directly compare a district in Kenya to one in Colombia

### H3 hexagons solve this

- **Uniform area** (~36 km² at level 6) — every hexagon is comparable
- **No edge effects** — all neighbors are equidistant (unlike squares)
- **Hierarchical** — you can aggregate up or drill down
- **Globally consistent** — same grid everywhere on Earth

Let’s see the difference visually.

In [ ]:
# Fetch boundaries and hex data for a small country to illustrate
ISO3 = "LKA"  # Sri Lanka
boundaries = client.fetch_admin_boundaries(ISO3, "ADM2")

# Get population at hex level
hex_data = client.get_summary(
    gdf=boundaries,
    spatial_join_method="centroid",
    fields=["sum_pop_2025"],
    geometry="polygon"
)

if isinstance(hex_data["geometry"].iloc[0], str):
    hex_data["geometry"] = hex_data["geometry"].apply(lambda g: from_geojson(g))
hex_gdf = gpd.GeoDataFrame(hex_data, geometry="geometry", crs="EPSG:4326")

import numpy as np

def round_bins(series, k=7):
    """Compute quantile breaks and round them to clean numbers."""
    quantiles = np.linspace(0, 1, k + 1)[1:-1]  # exclude 0 and 1
    breaks = series.dropna().quantile(quantiles).tolist()
    rounded = []
    for b in breaks:
        if b < 100:
            rounded.append(round(b, -1))        # round to 10s
        elif b < 1000:
            rounded.append(round(b, -2))         # round to 100s
        else:
            rounded.append(round(b, -3))         # round to 1000s
    # Deduplicate while preserving order
    seen = set()
    unique = []
    for r in rounded:
        if r not in seen and r > 0:
            seen.add(r)
            unique.append(int(r))
    return unique

POP_BINS = round_bins(hex_gdf["sum_pop_2025"])
POP_LABELS = [f"< {POP_BINS[0]}"] + \
             [f"{POP_BINS[i]}, {POP_BINS[i+1]}" for i in range(len(POP_BINS)-1)] + \
             [f"> {POP_BINS[-1]}"]

fig, ax = plt.subplots(figsize=(8, 6))
hex_gdf.plot(
    column="sum_pop_2025", cmap="YlOrRd",
    scheme="userdefined", classification_kwds={"bins": POP_BINS},
    legend=True, linewidth=0, ax=ax,
    legend_kwds={"title": "Population", "loc": "center left", "bbox_to_anchor": (1, 0.5),
                 "fontsize": 9, "labels": POP_LABELS}
)
boundaries.boundary.plot(ax=ax, color="black", linewidth=0.3, alpha=0.4)

ax.set_title("Sri Lanka - Population by H3 Hexagon", fontsize=12)
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Side-by-side comparison: admin boundaries vs hexagons
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: admin boundaries (aggregated)
agg_data = client.get_aggregate(
    gdf=boundaries,
    spatial_join_method="centroid",
    fields=["sum_pop_2025"],
    aggregation_type="sum"
)

AGG_BINS = round_bins(agg_data["sum_pop_2025"])
AGG_LABELS = [f"< {AGG_BINS[0]}"] + \
             [f"{AGG_BINS[i]}, {AGG_BINS[i+1]}" for i in range(len(AGG_BINS)-1)] + \
             [f"> {AGG_BINS[-1]}"]

agg_data.plot(
    column="sum_pop_2025", cmap="YlOrRd",
    scheme="userdefined", classification_kwds={"bins": AGG_BINS},
    legend=True, edgecolor="black", linewidth=0.3, ax=axes[0],
    legend_kwds={"title": "Population", "loc": "center left", "bbox_to_anchor": (1, 0.5),
                 "fontsize": 8, "labels": AGG_LABELS}
)
axes[0].set_title("Admin Boundaries (ADM2)", fontsize=12)
axes[0].axis("off")

# Right: hexagons (reuse POP_BINS from previous cell)
hex_gdf.plot(
    column="sum_pop_2025", cmap="YlOrRd",
    scheme="userdefined", classification_kwds={"bins": POP_BINS},
    legend=True, linewidth=0, ax=axes[1],
    legend_kwds={"title": "Population", "loc": "center left", "bbox_to_anchor": (1, 0.5),
                 "fontsize": 8, "labels": POP_LABELS}
)
axes[1].set_title("H3 Hexagons (Level 6)", fontsize=12)
axes[1].axis("off")

fig.suptitle("Sri Lanka - Population: Admin vs Hexagon View", fontsize=14, ha="center")
plt.tight_layout()
plt.show()

Notice how the hexagon view reveals **spatial patterns within districts** that the admin view hides. Many districts look uniformly dense in the admin view, but the hexagons show a more detailed picture of where population concentrates.

---
## Part 2: Discover What’s Available (5 min)

The API is **self-describing**. You don’t need to memorize field names — you can browse topics and fields programmatically.

In [ ]:
# What broad topics/datasets are available?
topics = client.get_topics()
topics

In [ ]:
# What specific fields can I query?
fields = client.get_fields()
print(f"Total available fields: {len(fields)}")
print("\nFirst 20 fields:")
fields[:20]

In [ ]:
# Get detailed metadata about a specific dataset
# (uses the STAC catalog under the hood)
pop_props = client.get_properties("world_pop")
pop_props.head(10)

### Discussion point

The STAC (SpatioTemporal Asset Catalog) metadata means every field has documented:
- **Source** (e.g., WorldPop, VIIRS, GHSL)
- **Units** and methodology
- **Temporal coverage**

This makes the data traceable and reproducible.

---
## Part 3: Hex-Level Data with `get_summary()` (10 min)

The primary way to access hex-level data is `get_summary()`. You provide an area of interest (admin boundaries) and it returns one row per hexagon.

### Spatial join methods

When querying, you choose how hexagons relate to your boundary:

| Method | Meaning | When to use |
|--------|---------|-------------|
| `centroid` | Only hexagons whose center falls inside the boundary | Most common — clean, no double-counting |
| `within` | Only hexagons fully contained in the boundary | Conservative estimate |
| `touches` | Any hexagon that overlaps the boundary at all | Inclusive — captures edges |

In [ ]:
# Fetch ADM2 boundaries for Rwanda
ISO3 = "RWA"
ADM = "ADM2"
rwa_boundaries = client.fetch_admin_boundaries(ISO3, ADM)
print(f"Number of districts: {len(rwa_boundaries)}")
rwa_boundaries.plot(figsize=(8, 8), edgecolor="black", linewidth=0.3, color="lightblue")
plt.title(f"{ISO3} – {ADM} Boundaries")
plt.axis("off")
plt.show()

In [ ]:
# Pick some population fields
pop_fields = ["sum_pop_2025", "sum_f_2025", "sum_m_2025"]

# Query hex-level data for all of Rwanda
rwa_hex_data = client.get_summary(
    gdf=rwa_boundaries,
    spatial_join_method="centroid",
    fields=pop_fields,
    geometry="polygon"  # include hex geometries for mapping
)

print(f"Rows returned: {len(rwa_hex_data)}")
rwa_hex_data.head()

In [ ]:
# Convert GeoJSON strings to proper geometries
if isinstance(rwa_hex_data["geometry"].iloc[0], str):
    rwa_hex_data["geometry"] = rwa_hex_data["geometry"].apply(lambda g: from_geojson(g))
rwa_hex_gdf = gpd.GeoDataFrame(rwa_hex_data, geometry="geometry", crs="EPSG:4326")

### Interactive map

Explore the hex-level data interactively. Hover to see population values, zoom and pan to explore.

In [ ]:
# Fill nulls as zero for display
for col in ["sum_pop_2025", "sum_f_2025", "sum_m_2025"]:
    rwa_hex_gdf[col] = rwa_hex_gdf[col].fillna(0)

# Pre-bin into categorical column so legend labels are clean
rwa_hex_gdf["pop_class"] = pd.cut(
    rwa_hex_gdf["sum_pop_2025"],
    bins=[0, 5000, 15000, 30000, 60000, 100000, 200000, float("inf")],
    labels=["< 5K", "5K - 15K", "15K - 30K", "30K - 60K", "60K - 100K", "100K - 200K", "> 200K"],
)

# Drop hexagons with zero population so they render as transparent
hex_nonzero = rwa_hex_gdf[rwa_hex_gdf["sum_pop_2025"] > 0].copy()

m = hex_nonzero.explore(
    column="pop_class",
    cmap="YlOrRd",
    categorical=True,
    legend=True,
    tiles="cartodbpositron",
    tooltip=["hex_id", "sum_pop_2025", "sum_f_2025", "sum_m_2025"],
    tooltip_kwds={"aliases": ["Hex ID", "Population", "Female", "Male"]},
    style_kwds={"weight": 0.5, "fillOpacity": 0.7},
    legend_kwds={"caption": "Population (2025)"},
    name="Population",
)

# Overlay admin boundaries
rwa_boundaries.boundary.explore(
    m=m,
    color="black",
    style_kwds={"weight": .3, "fillOpacity": 0},
    name="ADM2 Boundaries",
)

folium.LayerControl().add_to(m)
m

---
## Part 4: Aggregating Hex Data with `get_aggregate()` (5 min)

Sometimes you want the **district-level total** rather than individual hexagons. `get_aggregate()` sums (or averages, counts, etc.) hex values back up to each admin boundary.

In [ ]:
# Aggregate population to district level
rwa_agg = client.get_aggregate(
    gdf=rwa_boundaries,
    spatial_join_method="centroid",
    fields=["sum_pop_2025", "sum_viirs_ntl_2023"],
    aggregation_type="sum"
)

print(f"Districts returned: {len(rwa_agg)}")
rwa_agg[["NAM_2", "sum_pop_2025", "sum_viirs_ntl_2023"]].head(10)


In [ ]:
# Choropleth of aggregated population
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

rwa_agg.plot(
    column="sum_pop_2025", cmap="YlOrRd", scheme="quantiles", k=5,
    legend=True, edgecolor="black", linewidth=0.3, ax=axes[0],
    legend_kwds={"title": "Population", "loc": "lower left"}
)
axes[0].set_title("Total Population by District (sum)", fontsize=13)
axes[0].axis("off")

rwa_agg.plot(
    column="sum_viirs_ntl_2023", cmap="YlOrRd", scheme="quantiles", k=5,
    legend=True, edgecolor="black", linewidth=0.3, ax=axes[1],
    legend_kwds={"title": "NTL (sum)", "loc": "lower left"}
)
axes[1].set_title("Total Nighttime Lights by District (sum)", fontsize=13)
axes[1].axis("off")

plt.suptitle("Rwanda — Aggregated Hex Data at ADM2 Level", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()


---
## Part 5: Time Series at Hex Level (10 min)

This is where hexagons really shine. The SPI (Standardized Precipitation Index) dataset provides **monthly** drought data — you can track how conditions change over time at fine spatial resolution.

### SPI quick reference

| SPI Value | Meaning |
|-----------|--------|
| > 2.0 | Extremely wet |
| 1.0 to 2.0 | Moderately wet |
| -1.0 to 1.0 | Near normal |
| -1.0 to -1.5 | Moderate drought |
| -1.5 to -2.0 | Severe drought |
| < -2.0 | Extreme drought |

In [ ]:
# What time-series fields are available?
ts_fields = client.get_timeseries_fields()
print("Available time-series fields:", ts_fields)

In [ ]:
from space2stats_client.widgets import AOISelector

# Interactive map — draw a polygon or rectangle
aoi = AOISelector(center=(7.0, 80.0), zoom=7)  # Centered on Sri Lanka
aoi.display()

In [ ]:
# Query 12 months of SPI data for Sri Lanka
aoi_gdf = aoi.aoi.gdf

spi_data = client.get_timeseries(
    gdf=aoi_gdf,
    spatial_join_method="centroid",
    fields=["spi"],
    start_date="2024-01-01",
    end_date="2024-12-31",
    geometry="polygon"
)

print(f"Rows returned: {len(spi_data)}")
spi_data.head()

In [ ]:
# Prepare for visualization
spi_data["date"] = pd.to_datetime(spi_data["date"])

if isinstance(spi_data["geometry"].iloc[0], str):
    spi_data["geometry"] = spi_data["geometry"].apply(json.loads)
spi_data["geometry"] = spi_data["geometry"].apply(shape)
spi_gdf = gpd.GeoDataFrame(spi_data, geometry="geometry", crs="EPSG:4326")
spi_gdf["month"] = spi_gdf["date"].dt.strftime("%Y-%m")

In [ ]:
# Faceted map: drought conditions month by month
from plotnine import (
    ggplot, aes, geom_map, coord_fixed, facet_wrap,
    scale_fill_distiller, theme_void, theme, element_rect
)

(
    ggplot(spi_gdf)
    + geom_map(aes(fill="spi"), size=0)
    + scale_fill_distiller(type="div", palette="RdBu", name="SPI", limits=(-2, 2))
    + facet_wrap("month", ncol=4)
    + coord_fixed(expand=False)
    + theme_void()
    + theme(
        figure_size=(10, 8),
        plot_background=element_rect(fill="white"),
        panel_spacing=0.025
    )
)

---
## Part 6: Your Turn! (5 min)

### Challenge

Using what you've learned, try one of these:

**Option A — Drought hotspots:**
1. Pick a country and fetch its boundaries
2. Query SPI time-series data for a year
3. Find which hexagons had the **lowest average SPI** (worst drought)
4. Map those hexagons

**Option B — Urban vs rural comparison:**
1. Pick a country
2. Query both population and nighttime lights at hex level
3. Calculate a "lights per person" ratio per hexagon
4. Map it — where is there high population but low light (potential under-electrification)?

In [ ]:
# Your code here!


---
## Recap

| What we did | Method |
|---|---|
| Discovered available data | `get_topics()`, `get_fields()`, `get_properties()` |
| Compared admin vs hex views | `get_summary()` + `get_aggregate()` side by side |
| Got hex-level detail | `get_summary(gdf, spatial_join_method, fields, geometry)` |
| Aggregated to district level | `get_aggregate(gdf, spatial_join_method, fields, aggregation_type)` |
| Pulled monthly time series | `get_timeseries(gdf, fields, start_date, end_date)` |

### The full picture

| Scale | Question answered | S2S tool |
|-------|-------------------|----------|
| Country/Region | "What is the total population of this district?" | `get_aggregate()` |
| Hexagon | "Where within the district is the population?" | `get_summary()` |
| Hexagon + time | "How has drought changed month by month?" | `get_timeseries()` |